# [title] One-state model: the simulator

**Question.** Does `simulate_sitting` produce what the model says it should?

**Scope.** Step C1 of [#12](https://github.com/opherdonchin/MotorAdpatationSBI/issues/12).
The model is that of [decision 0005](../docs/decisions/0005-one-state-model-published-form.md),
written up in [docs/models/one_state.md](../docs/models/one_state.md); the simulator is in
[`simulators/one_state.py`](../simulators/one_state.py).

**Checks**, both against results that do not depend on this code:

1. *Without noise,* the simulated movements follow the closed-form learning curve: a
   geometric approach to a fixed point with vision, pure decay without.
2. *With noise,* in a long stretch of constant conditions, the standard deviation of the
   movements and the correlation between successive movements match the formulas of
   van der Vliet et al. (2018), equations 9 to 12.

In [ ]:
# [imports]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from simulators.one_state import simulate_sitting

In [ ]:
# [config] Root seed, example sitting, and the size of the steady-state check
ROOT_SEED = 229768114830082316116958060649188218563  # secrets.randbits(128)

# theta = (A, B, sigma_eta, sigma_epsilon), in units of the perturbation size.
THETA_EXAMPLE = (0.98, 0.1, 0.02, 0.15)  # close to the prior medians of decision 0005
EXAMPLE_BLOCKS = [("baseline", 40), ("+1", 80), ("no vision", 30), ("-1", 80), ("baseline", 40)]
TINY = 1e-12  # noise standard deviation that makes a sitting deterministic

# Parameter sets for the steady-state check: B small and large, A close to 1, and
# planning noise from small to large relative to execution noise.
STATIONARY_THETAS = [
    (0.98, 0.2, 0.02, 0.2),
    (0.98, 0.05, 0.03, 0.1),
    (0.95, 0.4, 0.05, 0.15),
    (0.995, 0.1, 0.01, 0.2),
]
N_SITTINGS = 2000  # independent sittings per parameter set and condition
T_BURN = 1000  # trials discarded first, until the start (x_1) no longer matters
T_KEEP = 800  # trials kept per sitting for the statistics
N_BATCHES = 20  # groups of sittings, for the Monte Carlo standard errors

In [ ]:
# [rng] One root generator, one stream per purpose
rng = np.random.default_rng(ROOT_SEED)
rng_example, rng_stationary = rng.spawn(2)

In [ ]:
# [plot-style] Colours and axes shared by the figures
BLUE, MUTED, GRID, SHADE = "#2a78d6", "#898781", "#e1e0d9", "#f0efec"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": "#52514e",
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 9, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "figure.constrained_layout.use": True,
})

## [model] The model

$$
x_1 \sim \mathcal N(0, \sigma_\eta^2), \qquad
y_t = x_t + \varepsilon_t, \qquad e_t = y_t + p_t, \qquad
x_{t+1} = A\, x_t - B\, v_t\, e_t + \eta_t
$$

$x_t$ is the plan, $y_t$ the movement, $p_t$ the perturbation, $e_t$ the error and $v_t$
the vision flag. The error is subtracted, so a learner compensates by moving to $-p$.
On a no-vision trial ($v_t = 0$) there is nothing to learn from and the plan only decays.

`simulate_sitting(rng, theta, p, v, size)` runs these equations forward, one trial at a
time, for `size` sittings at once.

In [ ]:
# [example-schedule] A hand-made schedule: one block of each kind
KIND = {"baseline": (0.0, 1), "+1": (1.0, 1), "-1": (-1.0, 1), "no vision": (0.0, 0)}
lengths = [n for _, n in EXAMPLE_BLOCKS]
p = np.repeat([KIND[k][0] for k, _ in EXAMPLE_BLOCKS], lengths)
v = np.repeat([KIND[k][1] for k, _ in EXAMPLE_BLOCKS], lengths)
trial = np.arange(1, len(p) + 1)

In [ ]:
# [example-sittings] Three simulated sittings and the noise-free curve
A, B, sigma_eta, sigma_epsilon = THETA_EXAMPLE
y_noisy = simulate_sitting(rng_example, THETA_EXAMPLE, p, v, size=(3,))
y_clean = simulate_sitting(rng_example, (A, B, TINY, TINY), p, v)

fig, ax = plt.subplots(figsize=(10, 3.6))
for start, stop in zip(*np.nonzero(np.diff(np.r_[1, v, 1]) != 0)[0].reshape(-1, 2).T):
    ax.axvspan(start + 0.5, stop + 0.5, color=SHADE, lw=0, label="no vision")
ax.step(trial, p, where="mid", color=MUTED, lw=1.5, label="perturbation $p_t$")
ax.plot(trial, y_noisy.T, ".", color=BLUE, ms=2.5, alpha=0.5)
ax.plot([], [], ".", color=BLUE, ms=5, label="movement $y_t$, three sittings")
ax.plot(trial, y_clean, color="#0b0b0b", lw=1.5, label="without noise")
ax.set_xlabel("trial")
ax.set_ylabel("perturbation units")
ax.set_title("A={}  B={}  $\\sigma_\\eta$={}  $\\sigma_\\varepsilon$={}".format(*THETA_EXAMPLE))
ax.grid(axis="y", color=GRID, lw=0.6)
ax.legend(loc="upper right", frameon=False, fontsize=8, ncols=2)
plt.show()

## [check-1] Check 1: the noise-free learning curve

Without noise, and with a constant perturbation $p$ and vision, the model has a fixed
point $y^* = -Bp / (1 - A + B)$, and the movement approaches it geometrically:
$y_{t+1} - y^* = (A - B)(y_t - y^*)$. Without vision the movement only decays:
$y_{t+1} = A\, y_t$. These closed forms are computed block by block, starting each block
where the previous one ended, and compared with the simulator run with negligible noise.

In [ ]:
# [check-noise-free] Closed-form curve, block by block, against the noise-free simulation
closed_form = []
y_start = 0.0  # x_1 = 0 without noise
for kind, n in EXAMPLE_BLOCKS:
    p_block, v_block = KIND[kind]
    k = np.arange(n + 1)  # n trials of the block, plus the first trial of the next
    if v_block:
        y_star = -B * p_block / (1 - A + B)
        curve = y_star + (y_start - y_star) * (A - B) ** k
    else:
        curve = y_start * A**k
    closed_form.append(curve[:-1])
    y_start = curve[-1]
closed_form = np.concatenate(closed_form)

max_abs_diff = np.abs(closed_form - y_clean).max()
assert max_abs_diff < 1e-9
print(f"largest difference between closed form and simulation: {max_abs_diff:.1e}")

## [check-2] Check 2: steady-state variability

Under a constant perturbation the movements settle into a stationary fluctuation around
the fixed point. Its size and its correlation from one trial to the next follow from the
model. With vision, writing $V_x = (\sigma_\eta^2 + B^2\sigma_\varepsilon^2)/(1-(A-B)^2)$
for the variance of the plan:

$$
\sigma_y^2 = \sigma_\varepsilon^2 + V_x, \qquad
R(1) = \frac{(A - B)\, V_x - B\, \sigma_\varepsilon^2}{\sigma_y^2}
$$

Without vision:

$$
\sigma_y^2 = \sigma_\varepsilon^2 + \frac{\sigma_\eta^2}{1 - A^2}, \qquad
R(1) = \frac{A\, \sigma_\eta^2 / (1 - A^2)}{\sigma_y^2}
$$

These are equations 9, 11 and 12 of van der Vliet et al. (2018) with the geometric sums
added up. For $R(1)$ with vision, the paper's equation 10 is printed differently:
$+B\sigma_\varepsilon^2$ in the numerator, and $A^k(A-B)^k$ in place of $(A-B)^{2k}$ in
the denominator. Both versions are computed below, as `R1 formula` (derived above) and
`R1 eq. 10 as printed`.

For each parameter set, `N_SITTINGS` sittings are simulated under $p = 0$, first with
vision and then without. The first `T_BURN` trials are discarded; the standard deviation
and the lag-1 correlation are computed from the rest, pooled over sittings. Standard
errors come from the spread between `N_BATCHES` groups of sittings.

In [ ]:
# [check-stationary] Simulated steady-state statistics against the formulas
rows = []
for theta in STATIONARY_THETAS:
    A_, B_, s_eta, s_eps = theta
    for vision in (1, 0):
        T = T_BURN + T_KEEP
        y = simulate_sitting(rng_stationary, theta, np.zeros(T), np.full(T, vision), size=(N_SITTINGS,))
        batches = y[:, T_BURN:].reshape(N_BATCHES, -1, T_KEEP)
        sd = batches.std(axis=(1, 2))
        r1 = np.array([np.corrcoef(b[:, :-1].ravel(), b[:, 1:].ravel())[0, 1] for b in batches])

        if vision:
            var_x = (s_eta**2 + B_**2 * s_eps**2) / (1 - (A_ - B_) ** 2)
            var_y = s_eps**2 + var_x
            r1_formula = ((A_ - B_) * var_x - B_ * s_eps**2) / var_y
            r1_printed = ((A_ - B_) * var_x + B_ * s_eps**2) / (
                s_eps**2 + (s_eta**2 + B_**2 * s_eps**2) / (1 - A_ * (A_ - B_))
            )
        else:
            var_y = s_eps**2 + s_eta**2 / (1 - A_**2)
            r1_formula = A_ * s_eta**2 / (1 - A_**2) / var_y
            r1_printed = np.nan  # equation 12 is the formula itself

        se_sd, se_r1 = sd.std(ddof=1) / np.sqrt(N_BATCHES), r1.std(ddof=1) / np.sqrt(N_BATCHES)
        rows.append({
            "theta (A, B, s_eta, s_eps)": theta,
            "vision": bool(vision),
            "sd simulated": sd.mean(),
            "sd formula": np.sqrt(var_y),
            "sd z": (sd.mean() - np.sqrt(var_y)) / se_sd,
            "R1 simulated": r1.mean(),
            "R1 formula": r1_formula,
            "R1 z": (r1.mean() - r1_formula) / se_r1,
            "R1 eq. 10 as printed": r1_printed,
        })
check_stationary = pd.DataFrame(rows)
check_stationary.round(4)